In [0]:
%pip install -U databricks-sdk databricks_vectorsearch 
%pip install --upgrade openai
dbutils.library.restartPython()

In [0]:
# Need to hard-code currently as config cannot run in serving endpoint
user_online_table_path = 'ml_shovakeemian.feip.user_pet_features_online'
vs_index_path = 'ml_shovakeemian.feip.cat_images_vs'
vs_index_endpoint = 'one-env-shared-endpoint-15'
image_table_path = 'ml_shovakeemian.feip.cat_images'
brand_table_path = 'ml_shovakeemian.feip.catfood_creatives'
image_gen_model_path = 'ml_shovakeemian.feip.cat_ad_image_gen'
endpoint_name = 'cat_ad_image_gen'

#AGENT BUILD

In [0]:
import base64
import mlflow
import pandas as pd
import os
import requests

from databricks.sdk import WorkspaceClient
from databricks.vector_search.index import VectorSearchIndex
from databricks.vector_search.client import VectorSearchClient
from io import BytesIO
from mlflow.deployments import get_deploy_client
from mlflow.entities import SpanType
from openai import OpenAI
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [0]:
os.environ['CLIENT_ID'] = dbutils.secrets.get("shovakeemian-scope", "shovakeemian-sp-client-id")
os.environ['CLIENT_SECRET'] = dbutils.secrets.get("shovakeemian-scope", "shovakeemian-sp-client-secret")
os.environ['OPENAI_KEY'] = dbutils.secrets.get("shovakeemian-scope", "openai-key")

In [0]:
class CAT_AGENT(mlflow.pyfunc.PythonModel):
    def __init__(self):
      self.session=requests.Session()

      self.oauth=None
      self.CLIENT_ID = os.environ.get("CLIENT_ID") 
      self.CLIENT_SECRET = os.environ.get("CLIENT_SECRET") 

      self.vsc=VectorSearchClient(    
          workspace_url="https://e2-demo-field-eng.cloud.databricks.com/",
          service_principal_client_id=self.CLIENT_ID,
          service_principal_client_secret=self.CLIENT_SECRET
      )
      
      self.online_table_url='https://80ad8fdc-e516-47cb-b709-85397e83bbf7.online-tables.cloud.databricks.com/api/2.0/workspace/1444828305810485/online/pgrest/ml_shovakeemian/user_pet_features_online'
      self.online_table_schema='feip'

      self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
      self.processor= CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")


    def load_context(self, context):
      self.api_key = os.environ.get("OPENAI_KEY")
      self.brand_image_path = context.artifacts.get("brand_image")

      with open(self.brand_image_path, "rb") as f:
        self.brand_image = f.read()


    def _ensure_openai_client(self):
      self.openai_client = OpenAI(api_key=self.api_key)
    

    @mlflow.trace(name="update_oauth_token")
    def update_oauth_token(self):
      url = 'https://e2-demo-field-eng.cloud.databricks.com/oidc/v1/token'
      data = {
          'grant_type': 'client_credentials',
          'client_id': self.CLIENT_ID,
          'client_secret': self.CLIENT_SECRET,
          'scope': 'all-apis',
          'authorization_details': '[{"type":"unity_catalog_permission","securable_type":"table","securable_object_name":"ml_shovakeemian.feip.user_pet_features_online","operation": "ReadOnlineView"}]'
      }

      response = requests.post(url, data=data)
      response.raise_for_status()
      self.oauth = response.json().get('access_token')
      return self.oauth


    @mlflow.trace(name="user_lookup", span_type=SpanType.RETRIEVER, attributes={"table": user_online_table_path})
    def _online_table_lookup(self, user):      
      headers = {
        "Authorization": f"Bearer {self.oauth}",
        "Accept-Profile": self.online_table_schema
        }
      query_params = {
        "select": "breed"
        ,"user_id": f"in.({user})"
        }
      try:
        response = self.session.get(self.online_table_url, headers=headers, params=query_params)
        cat_breed=response.json()[0]['breed']
      except:
        self.update_oauth_token()
        response = self.session.get(self.online_table_url, headers=headers, params=query_params)
        cat_breed=response.json()[0]['breed']
      
      cat_breed=cat_breed.replace('\u200b', '')
      return cat_breed


    @mlflow.trace(name="compute_text_embedding", span_type=SpanType.EMBEDDING, attributes={"model": "clip-vit-large-patch14"})
    def _get_text_embedding(self, text):
      """
      computes the text embedding for a given text.
      """
      # Want to change this to call the serving endpoint when ai_query can take params of a pyfunc.
      inputs = self.processor(text=text, return_tensors="pt", padding=True)
      text_features = self.model.get_text_features(**inputs)
      return text_features.detach().numpy().tolist()[0]


    @mlflow.trace(name="cat_image_vs_imageemb_lookup", span_type=SpanType.RETRIEVER, attributes={"model": "clip-vit-large-patch14", "vs_index": vs_index_path})
    def _vector_search_retrieval(self, query):
      text_embed_query=self._get_text_embedding(query)
      index = self.vsc.get_index(endpoint_name=vs_index_endpoint, index_name=vs_index_path)
      vs_output = index.similarity_search(columns=["id", "model_input"], query_vector=text_embed_query, num_results=3)
      return vs_output


    @mlflow.trace(name="cat_id_image_lookup", span_type=SpanType.RETRIEVER, attributes={"table": image_table_path})
    def _get_image(self, vs_output):
      """
      takes the output of vs and returns original byte array
      """
      image_base64 = [result[1] for result in vs_output['result']['data_array']][0]
      return image_base64
    
    
    @mlflow.trace(name="open ai image generation", span_type=SpanType.CHAT_MODEL, attributes={"table": image_table_path})
    def _openai_image_generation(self, cat_image, brand_image):
      self._ensure_openai_client()
      
      cat_image_bytes=base64.b64decode(cat_image)
      brand_image_bytes=bytes(brand_image)
      prompt=prompt = """
        create an advertisement for cat food with the images provided. Make the cat fromt he cat_image peak out the side of a bag of cat food found in the catfood_brand image. Make it as realistic as possible. Keep the cat in the image's original environemnt and try and incorporate the background. If there are multiple cats in the cat_image, make sure to include both; but if there is just one cat in the image, do not generate an image of a second one. 
      """
      result = self.openai_client.images.edit(
          model="gpt-image-1",
          image=[("cat_image.png", cat_image_bytes, "image/jpeg"), ("catfood_brand.png", brand_image_bytes, "image/png")],
          prompt=prompt
      )
      image_base64 = result.data[0].b64_json
      return (image_base64)


    @mlflow.trace(name="quickstart-agent")
    def predict(self, context, model_input):
      if isinstance(model_input, pd.DataFrame):
        query = model_input["model_input"].iloc[0]
      elif isinstance(model_input, dict):
        query = model_input.get("model_input", "")
      elif isinstance(model_input, str):
        query = model_input
      else:
        raise ValueError("Unsupported input type")

      vs_output=self._vector_search_retrieval(query=query)
      cat_image=self._get_image(vs_output)
      final_ad=self._openai_image_generation(cat_image=cat_image, brand_image=self.brand_image)

      return query, cat_image, final_ad

In [0]:
agent=CAT_AGENT()

class DummyContext:
    artifacts = {
        "brand_image": "/Volumes/ml_shovakeemian/feip/catfood_brand_creatives/BricksV3.png"
    }
agent.load_context(DummyContext)

In [0]:
model_input, cat_image, final_ad=agent.predict(model_input='black cat in the forest', context=None)

In [0]:
print(model_input)

In [0]:
Image.open(BytesIO(base64.b64decode(cat_image)))

In [0]:
Image.open(BytesIO(base64.b64decode(final_ad)))

In [0]:
# troublesome cats
# ids=[4544, 2862] #deleted April 28
# ids=[3273, 2139]

In [0]:
# %sql
# delete from ml_shovakeemian.feip.cat_images
# where id in (4544, 2862);

In [0]:
# %sql
# delete from ml_shovakeemian.feip.cat_images_embeddings
# where id in (4544, 2862);

## Deploy

In [0]:
import pandas as pd

example_input = pd.DataFrame({
    "model_input": ["black cat in the forest"]
})

In [0]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_schema = Schema([
    ColSpec("string", "model_input")
])

output_schema = Schema([
    ColSpec("string", "model_input"),
    ColSpec("string", "cat_image"),
    ColSpec("string", "final_ad"),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex

resources = [DatabricksVectorSearchIndex(index_name=vs_index_path)]

with mlflow.start_run():
  mlflow.pyfunc.log_model(
    "agent", 
    python_model=CAT_AGENT(),
    resources=resources,
    signature=signature,
    input_example=example_input,
    artifacts={
      "brand_image": "/Volumes/ml_shovakeemian/feip/catfood_brand_creatives/BricksV3.png"
    },
  )

In [0]:
run_id = mlflow.last_active_run().info.run_id

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=f"runs:/{run_id}/agent", name=image_gen_model_path
)

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
# response = client.create_endpoint(
#     name=endpoint_name,
#     config={
#         "served_entities": [
#             {
#                 "name": endpoint_name,
#                 "entity_name": image_gen_model_path,
#                 "entity_version": uc_registered_model_info.version,
#                 "workload_size": "Small",
#                 "scale_to_zero_enabled": True,
#                 "environment_vars": {
#                     "CLIENT_ID": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-id}}",
#                     "CLIENT_SECRET": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-secret}}",
#                     "OPENAI_KEY": "{{secrets/shovakeemian-scope/openai-key}}",
#                 }
#             }
#         ],
#     }
# )

# Update the existing endpoint
response = client.update_endpoint(
    endpoint=endpoint_name, # existing endpoint name
    config={
        "served_entities": [
            {
                "name": endpoint_name,
                "entity_name": image_gen_model_path,
                "entity_version": uc_registered_model_info.version,
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
                "environment_vars": {
                    "CLIENT_ID": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-id}}",
                    "CLIENT_SECRET": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-secret}}",
                    "OPENAI_KEY": "{{secrets/shovakeemian-scope/openai-key}}",
                }
            }
        ]
    }
)

print(response)